# COSMoS_BC_NCIt_Compare — Cross-Source Validation

Compare COSMoS Biomedical Concept definitions and synonyms against the
authoritative NCIt source (via the SDTM Test Identity reference file).

## Purpose

The COSMoS BC export carries its own `BC_Definition` and `BC_Synonyms`.
The SDTM Test Identity file carries `NCIt_Definition` and `NCIt_Synonyms`
from the NCIt Thesaurus. Both reference the same NCIt concepts via `NCIt_Code`.
This notebook surfaces where they diverge.

## Inputs

| File | Track | Role |
|---|---|---|
| `interim/COSMoS_BC_DSS.xlsx` | cosmos-bc-dss | The thing being validated |
| `machine_actionable/SDTM_Test_Identity.xlsx` | sdtm-test-codes | NCIt reference |

## Output

`reports/COSMoS_BC_NCIt_Compare.xlsx` — comparison report with:

| Sheet | Content |
|---|---|
| README | Provenance, check descriptions, summary statistics |
| Compare_Report | Summary: one row per check (same pattern as QC report) |
| Def_Differ | Rows where definitions diverge (BC-level) |
| Syn_Differ | Rows where synonym sets diverge (BC-level, classified by pattern) |
| No_Green_Match | BCs with NCIt_Code that has no match in SDTM CT test codes |

## Scope

Comparison runs at **BC level** — deduplicated on `NCIt_Code`.

Filtered to **subject-level Findings BCs only**:

- `Domain_Class == 'Findings'` — the observation class where TESTCD/TEST is the topic variable
- `BC_Scope != 'Study'` — excludes TS parameter BCs (trial-level metadata)
- `Row_Type != 'BC_only_hierarchy'` — excludes structural grouping nodes
- `NCIt_Code` populated — required for the join

Events and Interventions domains use different topic variables (--TERM, --TRT)
with no corresponding test code codelists — comparison against the green file
is not meaningful for those.

Within the Findings scope, some BCs have TESTCDs from **non-extensible**
instrument-specific codelists (PRO/ClinRO scales). These are not in the green
file, which covers only extensible domain-level test codes. The notebook
documents these as a structural boundary, not a gap.

## Pipeline Position

```
cosmos-bc-dss/interim/COSMoS_BC_DSS.xlsx  ──────────┐
                                                      ├── this notebook
sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx ──┘
                                                      │
                             reports/COSMoS_BC_NCIt_Compare.xlsx
```


## 1. Setup


In [1]:
import pandas as pd
import re
from pathlib import Path
from datetime import datetime
from collections import Counter
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('COSMoS BC NCIt COMPARE — CROSS-SOURCE VALIDATION')
print(f'Run: {datetime.now():%Y-%m-%d %H:%M}')
print()

# ── Paths ──
BASE_DIR = Path('..')              # cosmos-bc-dss/
REPO_ROOT = BASE_DIR / '..'       # cdisc-for-ai/

YELLOW_FILE = BASE_DIR / 'interim' / 'COSMoS_BC_DSS.xlsx'
GREEN_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'

REPORTS_DIR = BASE_DIR / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FILE = REPORTS_DIR / 'COSMoS_BC_NCIt_Compare.xlsx'

for f, label in [(YELLOW_FILE, 'Yellow (COSMoS)'), (GREEN_FILE, 'Green (NCIt)')]:
    if f.exists():
        print(f'  {label}: {f}')
    else:
        raise FileNotFoundError(f'{label} file not found: {f}')

print(f'  Output: {REPORT_FILE}')


COSMoS BC NCIt COMPARE — CROSS-SOURCE VALIDATION
Run: 2026-03-12 10:46

  Yellow (COSMoS): ../interim/COSMoS_BC_DSS.xlsx
  Green (NCIt): ../../sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  Output: ../reports/COSMoS_BC_NCIt_Compare.xlsx


## 2. Load Input Files


In [2]:
yellow = pd.read_excel(YELLOW_FILE, sheet_name='BC_DSS', dtype=str).fillna('')
green = pd.read_excel(GREEN_FILE, sheet_name='Test Codes', dtype=str).fillna('')

print(f'Yellow: {len(yellow):,} rows x {len(yellow.columns)} columns')
print(f'Green:  {len(green):,} rows x {len(green.columns)} columns')


Yellow: 1,545 rows x 34 columns
Green:  5,781 rows x 9 columns


## 3. Filter to Comparable BCs

Subject-level Findings BCs only. Exclude hierarchy nodes, rows without NCIt_Code,
Events/Interventions domains, and Study-scope (TS) BCs.
Deduplicate to BC level — a single BC may have multiple DSS rows,
all carrying the same definition and synonyms.


In [3]:
# Step 1: basic structural filters
has_ncit = yellow[
    (yellow['NCIt_Code'] != '') &
    (yellow['Row_Type'] != 'BC_only_hierarchy')
].copy()
n_all_non_hier = has_ncit.drop_duplicates(subset='NCIt_Code').shape[0]
print(f'All non-hierarchy BCs with NCIt_Code: {n_all_non_hier:,}')

# Step 2: scope filter — subject-level Findings only
comparable = has_ncit[
    (has_ncit['Domain_Class'] == 'Findings') &
    (has_ncit['BC_Scope'] != 'Study')
].copy()

# Report what was excluded and why
excluded = has_ncit[~has_ncit.index.isin(comparable.index)]
excl_bc = excluded.drop_duplicates(subset='NCIt_Code')
print(f'\nExcluded from comparison scope: {len(excl_bc):,} BCs')
print(f'  By Domain_Class: {excl_bc["Domain_Class"].value_counts().to_dict()}')
n_study = has_ncit[(has_ncit['Domain_Class'] == 'Findings') & (has_ncit['BC_Scope'] == 'Study')].drop_duplicates('NCIt_Code').shape[0]
if n_study > 0:
    print(f'  Study-scope Findings (TS parameters): {n_study}')

print(f'\nIn comparison scope: {comparable.drop_duplicates("NCIt_Code").shape[0]:,} BCs')
print(f'  Row types: {comparable["Row_Type"].value_counts().to_dict()}')

# Deduplicate to BC level — keep first row per NCIt_Code
# (BC_Definition and BC_Synonyms are identical across DSS rows for same BC)
bc_level = comparable.drop_duplicates(subset='NCIt_Code').copy()
print(f'  Unique BCs (NCIt_Code): {len(bc_level):,}')


All non-hierarchy BCs with NCIt_Code: 894

Excluded from comparison scope: 363 BCs
  By Domain_Class: {'': 193, 'Special-Purpose': 133, 'Events': 27, 'Interventions': 10}

In comparison scope: 531 BCs
  Row types: {'DSS': 920}
  Unique BCs (NCIt_Code): 531


## 4. Join Green Reference

Left join from BC-level yellow to green on NCIt_Code.
Green provides `NCIt_Definition`, `NCIt_Synonyms`, `NCIt_Preferred_Term`.


In [4]:
GREEN_COLS = ['NCIt_Code', 'NCIt_Preferred_Term', 'NCIt_Synonyms', 'NCIt_Definition']

green_lookup = green[GREEN_COLS].drop_duplicates(subset='NCIt_Code', keep='first')
print(f'Green lookup: {len(green_lookup):,} unique NCIt codes')

bc = bc_level.merge(
    green_lookup,
    on='NCIt_Code',
    how='left',
    indicator=True
).fillna('')

n_matched = (bc['_merge'] == 'both').sum()
n_no_match = (bc['_merge'] == 'left_only').sum()
print(f'\nMatched to green:   {n_matched:,}  (BCs with SDTM CT test code)')
print(f'No green match:     {n_no_match:,}  (BCs not in SDTM CT test codelists)')

bc_matched = bc[bc['_merge'] == 'both'].copy()
bc_no_match = bc[bc['_merge'] == 'left_only'].copy()
bc = bc.drop(columns=['_merge'])


Green lookup: 5,781 unique NCIt codes

Matched to green:   372  (BCs with SDTM CT test code)
No green match:     159  (BCs not in SDTM CT test codelists)


## 5. Comparison Functions

Definitions: case-insensitive, whitespace-normalized string comparison.
Synonyms: semicolon-delimited (both files), parsed into sets, order-independent.


In [5]:
def normalize(text):
    """Normalize for comparison: lowercase, collapse whitespace, strip."""
    if text is None or (isinstance(text, float) and pd.isna(text)) or str(text).strip() == '' or str(text).strip().lower() == 'nan':
        return ''
    return re.sub(r'\s+', ' ', str(text).strip().lower())


def normalize_synonym_set(text):
    """Parse synonyms into a normalized frozenset.
    Handles both semicolon (;) and pipe (|) delimiters for robustness."""
    if text is None or (isinstance(text, float) and pd.isna(text)) or str(text).strip() == '' or str(text).strip().lower() == 'nan':
        return frozenset()
    parts = [normalize(s) for s in re.split(r'[;|]', str(text)) if s.strip()]
    return frozenset(parts)


def compare_definitions(green_def, yellow_def):
    """Returns (flag, green_text, yellow_text)."""
    g = normalize(green_def)
    y = normalize(yellow_def)
    if not g and not y:
        return 'Both_Empty'
    if not g:
        return 'Green_Empty'
    if not y:
        return 'Yellow_Empty'
    if g == y:
        return 'Match'
    return 'Differ'


def compare_synonyms(green_syn, yellow_syn):
    """Returns (flag, pattern, detail)."""
    g = normalize_synonym_set(green_syn)
    y = normalize_synonym_set(yellow_syn)
    if not g and not y:
        return ('Both_Empty', '', '')
    if not g:
        return ('Green_Empty', '', '')
    if not y:
        return ('Yellow_Empty', '', '')
    if g == y:
        return ('Match', '', '')

    only_green = g - y
    only_yellow = y - g

    if only_green and not only_yellow:
        pattern = 'Green_Superset'
    elif only_yellow and not only_green:
        pattern = 'Yellow_Superset'
    else:
        pattern = 'Both_Unique'

    parts = []
    if only_green:
        parts.append(f'GREEN_ONLY: {" ; ".join(sorted(only_green))}')
    if only_yellow:
        parts.append(f'YELLOW_ONLY: {" ; ".join(sorted(only_yellow))}')

    return ('Differ', pattern, ' ||| '.join(parts))

print('Comparison functions defined.')


Comparison functions defined.


## 6. Run Comparisons


In [6]:
# Definition comparison — only on matched rows
bc_matched['Def_Flag'] = bc_matched.apply(
    lambda row: compare_definitions(row['NCIt_Definition'], row['BC_Definition']),
    axis=1
)

print('=== Definition Comparison ===')
print(f'Scope: {len(bc_matched):,} BCs with green match')
print(bc_matched['Def_Flag'].value_counts().to_string())

# Synonym comparison — only on matched rows
syn_results = bc_matched.apply(
    lambda row: compare_synonyms(row['NCIt_Synonyms'], row['BC_Synonyms']),
    axis=1,
    result_type='expand'
)
bc_matched['Syn_Flag'] = syn_results[0]
bc_matched['Syn_Pattern'] = syn_results[1]
bc_matched['Syn_Detail'] = syn_results[2]

print()
print('=== Synonym Comparison ===')
print(f'Scope: {len(bc_matched):,} BCs with green match')
print(bc_matched['Syn_Flag'].value_counts().to_string())
print()
print('Differ patterns:')
syn_differs = bc_matched[bc_matched['Syn_Flag'] == 'Differ']
print(syn_differs['Syn_Pattern'].value_counts().to_string())


=== Definition Comparison ===
Scope: 372 BCs with green match
Def_Flag
Match     369
Differ      3

=== Synonym Comparison ===
Scope: 372 BCs with green match
Syn_Flag
Match           242
Differ          128
Yellow_Empty      2

Differ patterns:
Syn_Pattern
Green_Superset     78
Yellow_Superset    25
Both_Unique        25


## 7. Build Comparison Report

Same structure as the QC report: one summary row per check.


In [7]:
compare_results = []

def cmp_flag(check_id, severity, description, count, detail=''):
    compare_results.append({
        'Check': check_id, 'Severity': severity,
        'Description': description, 'Count': count, 'Detail': detail
    })
    print(f'  [{severity:7s}] {check_id} {description} -- {count}')

def cmp_info(check_id, description, count, detail=''):
    cmp_flag(check_id, 'INFO', description, count, detail)

print('=' * 70)
print('CROSS-SOURCE COMPARISON REPORT')
print('=' * 70)

# CMP-01: Coverage — BCs matched to green
cmp_info('CMP-01', f'BCs with NCIt_Code matched to SDTM CT (of {len(bc_level)} comparable)',
         n_matched)

# CMP-02: Coverage — BCs not in green (instrument-specific TESTCDs)
# Cross-reference: some no-match BCs may be QC-14 cases where TESTCD_NCIt != NCIt_Code.
# The join uses NCIt_Code (BC identity), but the green file indexes by TESTCD's NCIt code.
# When these diverge, the join misses even though the TESTCD exists in green under a different code.
qc14_in_no_match = bc_no_match[
    (bc_no_match['TESTCD_NCIt'] != '') &
    (bc_no_match['TESTCD_NCIt'] != bc_no_match['NCIt_Code'])
]
n_qc14 = len(qc14_in_no_match)
domain_detail = f'Domains: {bc_no_match["Domain"].value_counts().head(5).to_dict()}'
if n_qc14 > 0:
    qc14_detail = ' | '.join(f"{r['TESTCD']} (BC={r['NCIt_Code']}, TESTCD={r['TESTCD_NCIt']})" for _, r in qc14_in_no_match.iterrows())
    domain_detail += f' | QC-14 related ({n_qc14}): {qc14_detail}'
cmp_info('CMP-02', 'Findings BCs not in green (instrument-specific or pending CT)',
         n_no_match, domain_detail)

# CMP-03: Definition match
n_def_match = (bc_matched['Def_Flag'] == 'Match').sum()
cmp_info('CMP-03', f'Definitions match (of {n_matched} matched BCs)',
         int(n_def_match))

# CMP-04: Definition differ
n_def_differ = (bc_matched['Def_Flag'] == 'Differ').sum()
if n_def_differ > 0:
    def_diff_rows = bc_matched[bc_matched['Def_Flag'] == 'Differ']
    detail = ' | '.join(f"{r['TESTCD']} ({r['NCIt_Code']})" for _, r in def_diff_rows.iterrows())
    cmp_flag('CMP-04', 'WARNING', 'Definitions differ between COSMoS and NCIt',
             int(n_def_differ), detail)
else:
    cmp_flag('CMP-04', 'PASS', 'No definition divergences', 0)

# CMP-05: Synonym match
n_syn_match = (bc_matched['Syn_Flag'] == 'Match').sum()
cmp_info('CMP-05', f'Synonym sets match (of {n_matched} matched BCs)',
         int(n_syn_match))

# CMP-06: Synonyms — green superset
n_green_super = (bc_matched['Syn_Pattern'] == 'Green_Superset').sum()
cmp_info('CMP-06', 'Synonyms: NCIt has more than COSMoS (green superset)',
         int(n_green_super))

# CMP-07: Synonyms — yellow superset
n_yellow_super = (bc_matched['Syn_Pattern'] == 'Yellow_Superset').sum()
if n_yellow_super > 0:
    cmp_flag('CMP-07', 'INFO', 'Synonyms: COSMoS has terms not in NCIt (yellow superset)',
             int(n_yellow_super))
else:
    cmp_flag('CMP-07', 'PASS', 'No COSMoS-only synonyms', 0)

# CMP-08: Synonyms — both have unique
n_both_unique = (bc_matched['Syn_Pattern'] == 'Both_Unique').sum()
if n_both_unique > 0:
    cmp_flag('CMP-08', 'INFO', 'Synonyms: both sources have unique terms',
             int(n_both_unique))
else:
    cmp_flag('CMP-08', 'PASS', 'No bidirectional synonym divergence', 0)

# CMP-09: Synonyms — yellow empty
n_syn_yellow_empty = (bc_matched['Syn_Flag'] == 'Yellow_Empty').sum()
if n_syn_yellow_empty > 0:
    cmp_info('CMP-09', 'COSMoS BC_Synonyms empty where NCIt has synonyms',
             int(n_syn_yellow_empty))

print()
print(f'Total checks: {len(compare_results)}')


CROSS-SOURCE COMPARISON REPORT
  [INFO   ] CMP-01 BCs with NCIt_Code matched to SDTM CT (of 531 comparable) -- 372
  [INFO   ] CMP-02 Findings BCs not in green (instrument-specific or pending CT) -- 159
  [INFO   ] CMP-03 Definitions match (of 372 matched BCs) -- 369
  [WARNING] CMP-04 Definitions differ between COSMoS and NCIt -- 3
  [INFO   ] CMP-05 Synonym sets match (of 372 matched BCs) -- 242
  [INFO   ] CMP-06 Synonyms: NCIt has more than COSMoS (green superset) -- 78
  [INFO   ] CMP-07 Synonyms: COSMoS has terms not in NCIt (yellow superset) -- 25
  [INFO   ] CMP-08 Synonyms: both sources have unique terms -- 25
  [INFO   ] CMP-09 COSMoS BC_Synonyms empty where NCIt has synonyms -- 2

Total checks: 9


## 8. Prepare Detail Sheets


In [8]:
# Definition differs — full detail
def_differ_df = bc_matched[bc_matched['Def_Flag'] == 'Differ'][
    ['NCIt_Code', 'TESTCD', 'BC_Name', 'BC_Definition', 'NCIt_Definition']
].copy()
print(f'Def_Differ sheet: {len(def_differ_df)} rows')

# Synonym differs — full detail with pattern classification
syn_differ_df = bc_matched[bc_matched['Syn_Flag'] == 'Differ'][
    ['NCIt_Code', 'TESTCD', 'BC_Name', 'Syn_Pattern', 'BC_Synonyms', 'NCIt_Synonyms', 'Syn_Detail']
].copy()
print(f'Syn_Differ sheet: {len(syn_differ_df)} rows')

# No green match — BCs that couldn't be compared
no_match_df = bc_no_match[
    ['NCIt_Code', 'TESTCD', 'BC_Name', 'BC_Type', 'Row_Type', 'Domain', 'Categories', 'Domains_with_DS']
].copy()
print(f'No_Green_Match sheet: {len(no_match_df)} rows')
print(f'  Domains: {no_match_df["Domain"].value_counts().head(5).to_dict()}')


Def_Differ sheet: 3 rows
Syn_Differ sheet: 128 rows
No_Green_Match sheet: 159 rows
  Domains: {'RS': 114, 'FT': 23, 'QS': 13, 'VS': 2, 'IE': 2}


## 9. Write Report


In [9]:
# ── Styles (matching Validate notebook pattern) ──
HEADER_FONT = Font(name='Arial', bold=True, size=10)
HEADER_FONT_WHITE = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')
thin_border = Border(
    left=Side(style='thin', color='D0D0D0'),
    right=Side(style='thin', color='D0D0D0'),
    top=Side(style='thin', color='D0D0D0'),
    bottom=Side(style='thin', color='D0D0D0'),
)

HEADER_BG = PatternFill('solid', fgColor='4472C4')
SECTION_BG = PatternFill('solid', fgColor='D6E4F0')

SEV_FILLS = {
    'WARNING': PatternFill('solid', fgColor='FFF2CC'),
    'INFO':    PatternFill('solid', fgColor='E2EFDA'),
    'PASS':    PatternFill('solid', fgColor='FFFFFF'),
}
SEV_FONTS = {
    'WARNING': Font(name='Arial', bold=True, size=10, color='BF8F00'),
    'INFO':    Font(name='Arial', size=10, color='548235'),
    'PASS':    Font(name='Arial', size=10, color='808080'),
}

# ── README data ──
readme_data = [
    ['SECTION', 'ITEM', 'DESCRIPTION'],
    ['PROVENANCE', 'Generated', f'{datetime.now():%Y-%m-%d %H:%M}'],
    ['', 'Yellow source', YELLOW_FILE.name],
    ['', 'Green source', GREEN_FILE.name],
    ['', 'Notebook', 'COSMoS_BC_NCIt_Compare.ipynb'],
    ['', '', ''],
    ['SCOPE', 'What', 'Cross-source comparison of COSMoS BC definitions and synonyms against NCIt'],
    ['', 'Filter', 'Subject-level Findings BCs only (Domain_Class=Findings, BC_Scope!=Study)'],
    ['', 'Rationale', 'Findings is the observation class where TESTCD/TEST is the topic variable — the structural overlap with the green file'],
    ['', 'Level', 'BC level — deduplicated on NCIt_Code'],
    ['', 'Join key', 'NCIt_Code (present in both files)'],
    ['', 'BCs compared', f'{len(bc_level)} in scope, {n_matched} matched to green, {n_no_match} instrument-specific (no green match)'],
    ['', '', ''],
    ['CHECKS', 'CMP-01', 'Coverage: BCs matched to SDTM CT test codes'],
    ['', 'CMP-02', 'Coverage: BCs with NCIt_Code not in SDTM CT'],
    ['', 'CMP-03', 'Definition match count'],
    ['', 'CMP-04', 'Definitions that differ between COSMoS and NCIt'],
    ['', 'CMP-05', 'Synonym set match count'],
    ['', 'CMP-06', 'Synonyms: NCIt superset (has more than COSMoS)'],
    ['', 'CMP-07', 'Synonyms: COSMoS superset (has terms not in NCIt)'],
    ['', 'CMP-08', 'Synonyms: both have unique terms'],
    ['', 'CMP-09', 'COSMoS synonyms empty where NCIt has values'],
    ['', '', ''],
    ['COMPARISON LOGIC', 'Definitions', 'Case-insensitive, whitespace-normalized string comparison'],
    ['', 'Synonyms', 'Semicolon-delimited, parsed into sets, order-independent comparison'],
    ['', 'Green_Superset', 'NCIt has all COSMoS synonyms plus additional ones'],
    ['', 'Yellow_Superset', 'COSMoS has all NCIt synonyms plus additional ones'],
    ['', 'Both_Unique', 'Each source has synonyms the other does not'],
    ['', '', ''],
    ['SEVERITY', 'WARNING', 'Content divergence worth reviewing'],
    ['', 'INFO', 'Expected pattern, documented for awareness'],
    ['', 'PASS', 'Check passed'],
]

# ── Write ──
compare_df = pd.DataFrame(compare_results)

with pd.ExcelWriter(REPORT_FILE, engine='openpyxl') as writer:
    pd.DataFrame(readme_data[1:], columns=readme_data[0]).to_excel(
        writer, sheet_name='README', index=False)
    compare_df.to_excel(writer, sheet_name='Compare_Report', index=False)
    if len(def_differ_df) > 0:
        def_differ_df.to_excel(writer, sheet_name='Def_Differ', index=False)
    if len(syn_differ_df) > 0:
        syn_differ_df.to_excel(writer, sheet_name='Syn_Differ', index=False)
    if len(no_match_df) > 0:
        no_match_df.to_excel(writer, sheet_name='No_Green_Match', index=False)

print(f'Wrote {REPORT_FILE.name}')


Wrote COSMoS_BC_NCIt_Compare.xlsx


In [10]:
wb = load_workbook(REPORT_FILE)

# ── Format README ──
ws = wb['README']
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    for cell in row:
        cell.border = thin_border
        cell.alignment = WRAP
for cell in ws[1]:
    cell.font = HEADER_FONT_WHITE
    cell.fill = HEADER_BG
for row in ws.iter_rows(min_row=2):
    val = str(row[0].value) if row[0].value else ''
    if val and val == val.upper() and val.strip():
        for cell in row:
            cell.font = HEADER_FONT
            cell.fill = SECTION_BG
ws.column_dimensions['A'].width = 22
ws.column_dimensions['B'].width = 22
ws.column_dimensions['C'].width = 80

# ── Format Compare_Report ──
ws = wb['Compare_Report']
for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
    for cell in row:
        cell.border = thin_border
        cell.alignment = WRAP
for cell in ws[1]:
    cell.font = HEADER_FONT_WHITE
    cell.fill = HEADER_BG
ws.column_dimensions['A'].width = 10
ws.column_dimensions['B'].width = 12
ws.column_dimensions['C'].width = 60
ws.column_dimensions['D'].width = 8
ws.column_dimensions['E'].width = 80

sev_col_idx = [cell.value for cell in ws[1]].index('Severity') + 1
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    sev = row[sev_col_idx - 1].value or ''
    fill = SEV_FILLS.get(sev, PatternFill('solid', fgColor='FFFFFF'))
    font = SEV_FONTS.get(sev, Font())
    for cell in row:
        cell.fill = fill
    row[sev_col_idx - 1].font = font
ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions


def format_detail_sheet(ws, col_widths):
    """Apply standard formatting to a detail sheet."""
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
        for cell in row:
            cell.border = thin_border
            cell.alignment = WRAP
            cell.font = DATA_FONT
    for cell in ws[1]:
        cell.font = HEADER_FONT_WHITE
        cell.fill = HEADER_BG
    for ci, width in enumerate(col_widths, 1):
        ws.column_dimensions[get_column_letter(ci)].width = width
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions


# ── Format detail sheets ──
if 'Def_Differ' in wb.sheetnames:
    format_detail_sheet(wb['Def_Differ'], [12, 12, 35, 60, 60])

if 'Syn_Differ' in wb.sheetnames:
    format_detail_sheet(wb['Syn_Differ'], [12, 12, 35, 16, 50, 50, 60])

if 'No_Green_Match' in wb.sheetnames:
    format_detail_sheet(wb['No_Green_Match'], [12, 12, 35, 12, 18, 8, 25, 20])

wb.save(REPORT_FILE)
print(f'Formatted {REPORT_FILE.name}')


Formatted COSMoS_BC_NCIt_Compare.xlsx


## 10. Summary


In [11]:
print()
print('=' * 70)
print('COMPLETE: COSMoS_BC_NCIt_Compare.xlsx')
print('=' * 70)
print(f'''
Output: {REPORT_FILE}

Scope: subject-level Findings BCs
  In scope:              {len(bc_level):,}
  Out of scope:          {n_all_non_hier - len(bc_level):,}  (Events, Interventions, Study-scope, no Domain_Class)

Coverage:
  Matched to green:      {n_matched:,}
  No green match:        {n_no_match:,}  (instrument-specific TESTCDs)

Definitions ({n_matched} compared):
  Match:                 {int(n_def_match):,}
  Differ:                {int(n_def_differ):,}

Synonyms ({n_matched} compared):
  Match:                 {int(n_syn_match):,}
  Green superset:        {int(n_green_super):,}
  Yellow superset:       {int(n_yellow_super):,}
  Both unique:           {int(n_both_unique):,}
  Yellow empty:          {int(n_syn_yellow_empty):,}
''')



COMPLETE: COSMoS_BC_NCIt_Compare.xlsx

Output: ../reports/COSMoS_BC_NCIt_Compare.xlsx

Scope: subject-level Findings BCs
  In scope:              531
  Out of scope:          363  (Events, Interventions, Study-scope, no Domain_Class)

Coverage:
  Matched to green:      372
  No green match:        159  (instrument-specific TESTCDs)

Definitions (372 compared):
  Match:                 369
  Differ:                3

Synonyms (372 compared):
  Match:                 242
  Green superset:        78
  Yellow superset:       25
  Both unique:           25
  Yellow empty:          2

